# Data Cleaning & Preprocessing

## Steps
1. Load raw data and inspect shape / dtypes / nulls
2. Remove invalid prices (< 50,000 or > 20,000,000 EGP)
3. Handle mileage = 0 → flag as missing, impute with brand+model median later
4. Handle transmission = "0" → flag as missing, impute with mode
5. Remove exact duplicate listings (same title + year + mileage + price)
6. Standardize location names → extract governorate → assign tier (A / B / C)
7. IQR-based outlier removal per brand+model group (1.5× IQR)
8. Log row counts after each step (cleaning funnel)
9. Save cleaned dataset to `data/processed/`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

# Load raw data
RAW_PATH = "../data/raw/"
PROCESSED_PATH = "../data/processed/"

# df = pd.read_parquet(f"{RAW_PATH}cars_raw.parquet")
# print(f"Raw shape: {df.shape}")
# df.head()

## Step 1 — Initial Inspection

Check shape, dtypes, null counts, and sample rows before any cleaning.

In [ ]:
# TODO: Uncomment when data is available
# print(f"Shape: {df.shape}")
# print(f"\nDtypes:\n{df.dtypes}")
# print(f"\nNull counts:\n{df.isnull().sum()}")
# print(f"\nDescribe:\n{df.describe()}")
# df.sample(5)

## Step 2 — Price Filtering

Remove listings with unrealistic prices:
- **Floor:** < 50,000 EGP (likely errors or parts listings)
- **Ceiling:** > 20,000,000 EGP (likely errors or luxury outliers)

Track the cleaning funnel — log how many rows are dropped at each step.

In [ ]:
# cleaning_funnel = {"raw": len(df)}
#
# df = df[(df["price"] >= 50_000) & (df["price"] <= 20_000_000)]
# cleaning_funnel["after_price_filter"] = len(df)
# print(f"Dropped {cleaning_funnel['raw'] - cleaning_funnel['after_price_filter']} rows by price filter")

## Step 3 — Handle Invalid Mileage & Transmission

**Mileage = 0:** Not truly zero — flag as NaN, impute later with brand+model median.  
**Transmission = "0":** Data entry error — flag as NaN, impute with mode per brand+model.

In [ ]:
# df.loc[df["mileage"] == 0, "mileage"] = np.nan
# df.loc[df["transmission"] == "0", "transmission"] = np.nan
#
# print(f"Mileage NaN count: {df['mileage'].isna().sum()}")
# print(f"Transmission NaN count: {df['transmission'].isna().sum()}")

## Step 4 — Remove Duplicate Listings

De-duplicate on key columns: `title + year + mileage + price`.  
Keep the first occurrence (most recent scrape if data is time-ordered).

In [ ]:
# before = len(df)
# df = df.drop_duplicates(subset=["title", "year", "mileage", "price"], keep="first")
# cleaning_funnel["after_dedup"] = len(df)
# print(f"Dropped {before - len(df)} duplicate rows")

## Step 5 — Location Standardization

Egyptian locations are messy in listings. Strategy:
1. Strip whitespace, lowercase, remove diacritics
2. Map common variations → canonical governorate name (e.g., "masr el gdida" → "Cairo")
3. Assign **location tier**: A (Cairo, Giza, Alex), B (Mansoura, Tanta, etc.), C (everything else)

> The location mapping dictionary will be built during EDA exploration.

In [ ]:
# TODO: Build location mapping from EDA findings
# location_map = {}  # raw_location -> canonical_governorate
# tier_map = {}       # governorate -> tier (A/B/C)
#
# df["governorate"] = df["location"].map(location_map)
# df["location_tier"] = df["governorate"].map(tier_map)

## Step 6 — IQR Outlier Removal

Remove price outliers **within each brand+model group** (not globally):
- Compute Q1, Q3, IQR per group
- Remove rows where price < Q1 - 1.5×IQR or price > Q3 + 1.5×IQR
- Only apply to groups with ≥ 10 samples (skip rare models)

In [ ]:
# def remove_outliers_iqr(group, col="price", factor=1.5):
#     q1 = group[col].quantile(0.25)
#     q3 = group[col].quantile(0.75)
#     iqr = q3 - q1
#     return group[(group[col] >= q1 - factor * iqr) & (group[col] <= q3 + factor * iqr)]
#
# before = len(df)
# df = df.groupby(["brand", "model"], group_keys=False).apply(
#     lambda g: remove_outliers_iqr(g) if len(g) >= 10 else g
# )
# cleaning_funnel["after_outlier_removal"] = len(df)
# print(f"Dropped {before - len(df)} outlier rows")

## Step 7 — Cleaning Funnel Summary

Visualize how many rows survived each cleaning step.  
This is important for the GP report to show data quality awareness.

In [ ]:
# funnel_df = pd.DataFrame(
#     {"step": cleaning_funnel.keys(), "rows": cleaning_funnel.values()}
# )
# print(funnel_df)
#
# fig, ax = plt.subplots(figsize=(8, 4))
# ax.barh(funnel_df["step"], funnel_df["rows"])
# ax.set_xlabel("Row Count")
# ax.set_title("Data Cleaning Funnel")
# plt.tight_layout()
# plt.show()

## Step 8 — Save Cleaned Dataset

Save to `data/processed/cleaned.parquet` for the next notebook (03_feature_engineering).

In [ ]:
# df.to_parquet(f"{PROCESSED_PATH}cleaned.parquet", index=False)
# print(f"Saved cleaned dataset: {df.shape}")
# print(f"Cleaning rate: {len(df) / cleaning_funnel['raw'] * 100:.1f}% of rows retained")